# Create IRT groups on Junyi dataset

## Install IRT

In [ ]:
!pip install -q py-irt

import pyro
print("Pyro version:", pyro.__version__)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 760.7/760.7 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 258.0/258.0 kB 16.0 MB/s eta 0:00:00
Pyro version: 1.9.2


## Imports

In [ ]:
# Imports
import os
import random
import pandas as pd
import numpy as np
import torch

from py_irt.models.one_param_logistic import OneParamLog

# Reproducibility
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
pyro.set_rng_seed(SEED)

# Mount Google Drive
from google.colab import drive
drive.mount("/content/drive")

# Paths
DATA_DIR = "/content/drive/MyDrive/education-ml-research/Junyi"

DATA_PATH = os.path.join(
    DATA_DIR,
    "Log_Problem.csv"
)

ABILITY_GROUP_PATH = os.path.join(
    DATA_DIR,
    "student_irt_ability_groups.csv"
)

# Load dataset
junyi_df = pd.read_csv(DATA_PATH)

print("Dataset shape:", junyi_df.shape)
print("Number of students:", junyi_df["uuid"].nunique())
print("Number of skills:", junyi_df["ucid"].nunique())
print("Number of problems:", junyi_df["upid"].nunique())

Mounted at /content/drive
Dataset shape: (16217311, 14)
Number of students: 72758
Number of skills: 1326
Number of problems: 25785


## Prepare IRT data

In [ ]:

irt_df = junyi_df[
    ["uuid", "ucid", "is_correct"]
].copy()

# Convert correctness to numeric
irt_df["is_correct"] = (
    irt_df["is_correct"]
    .astype(int)
)

# Keep only valid binary responses
irt_df = irt_df[
    irt_df["is_correct"].isin([0, 1])
].copy()

irt_df = irt_df.reset_index(drop=True)

print("Valid IRT interactions:", len(irt_df))
print("Students:", irt_df["uuid"].nunique())
print("Skills:", irt_df["ucid"].nunique())

print("\nResponse counts:")
print(irt_df["is_correct"].value_counts())

Valid IRT interactions: 16217311
Students: 72758
Skills: 1326

Response counts:
is_correct
1    11412558
0     4804753
Name: count, dtype: int64


## Create the two IRT datasets

In [ ]:
# Primary dataset: all encounters
irt_all = junyi_df[
    ["uuid", "upid", "is_correct"]
].copy()

irt_all["is_correct"] = irt_all["is_correct"].astype(int)

irt_all = irt_all[
    irt_all["is_correct"].isin([0, 1])
].reset_index(drop=True)


# Robustness dataset: first encounter only
irt_first = (
    irt_all
    .drop_duplicates(
        subset=["uuid", "upid"],
        keep="first"
    )
    .reset_index(drop=True)
)

print("ALL ENCOUNTERS")
print("Observations:", len(irt_all))
print("Students:", irt_all["uuid"].nunique())
print("Problems:", irt_all["upid"].nunique())

print("\nFIRST ENCOUNTER ONLY")
print("Observations:", len(irt_first))
print("Students:", irt_first["uuid"].nunique())
print("Problems:", irt_first["upid"].nunique())

ALL ENCOUNTERS
Observations: 16217311
Students: 72758
Problems: 25785

FIRST ENCOUNTER ONLY
Observations: 13785094
Students: 72758
Problems: 25785


## Create function to prepare tensors

In [ ]:
def prepare_irt_data(data):

    student_ids = np.sort(
        data["uuid"].unique()
    )

    problem_ids = np.sort(
        data["upid"].unique()
    )

    student_to_idx = {
        student_id: i
        for i, student_id in enumerate(student_ids)
    }

    problem_to_idx = {
        problem_id: i
        for i, problem_id in enumerate(problem_ids)
    }

    models = torch.tensor(
        data["uuid"].map(student_to_idx).values,
        dtype=torch.long
    )

    items = torch.tensor(
        data["upid"].map(problem_to_idx).values,
        dtype=torch.long
    )

    responses = torch.tensor(
        data["is_correct"].values,
        dtype=torch.float32
    )

    return (
        models,
        items,
        responses,
        student_ids,
        problem_ids
    )

(
    models_all,
    items_all,
    responses_all,
    student_ids_all,
    problem_ids_all
) = prepare_irt_data(irt_all)


(
    models_first,
    items_first,
    responses_first,
    student_ids_first,
    problem_ids_first
) = prepare_irt_data(irt_first)


print("All encounters:", len(responses_all))
print("First encounters:", len(responses_first))

All encounters: 16217311
First encounters: 13785094


## Fit the primary IRT model

In [ ]:
device = "cpu"

models_all = models_all.to(device)
items_all = items_all.to(device)
responses_all = responses_all.to(device)

pyro.clear_param_store()

irt_model_all = OneParamLog(
    priors="hierarchical",
    device=device,
    num_items=len(problem_ids_all),
    num_models=len(student_ids_all),
    verbose=True
)

NUM_EPOCHS = 1000

irt_model_all.fit(
    models=models_all,
    items=items_all,
    responses=responses_all,
    num_epochs=NUM_EPOCHS
)

[epoch 0001] loss: 555154434.1293
[epoch 0101] loss: 8644536.8207
[epoch 0201] loss: 8668248.9994
[epoch 0301] loss: 8552416.7307
[epoch 0401] loss: 8373360.9535
[epoch 0501] loss: 8084497.0583
[epoch 0601] loss: 9672684.3556
[epoch 0701] loss: 8299641.7714
[epoch 0801] loss: 8358625.7170
[epoch 0901] loss: 8510612.4046
[epoch 1000] loss: 8460817.0069


## Extract student ability estimates from the primary IRT model

In [ ]:
param_store_all = pyro.get_param_store()

theta_all = (
    param_store_all["loc_ability"]
    .detach()
    .cpu()
    .numpy()
)

irt_scores_all = pd.DataFrame({
    "uuid": student_ids_all,
    "theta_all": theta_all
})

print("Number of students:", len(irt_scores_all))
print("\nTheta summary:")
print(irt_scores_all["theta_all"].describe())

print("\nMissing theta values:")
print(irt_scores_all["theta_all"].isna().sum())

print("\nInfinite theta values:")
print(np.isinf(irt_scores_all["theta_all"]).sum())

Number of students: 72758

Theta summary:
count    72758.000000
mean        -0.125281
std          1.147566
min         -6.856836
25%         -0.846577
50%         -0.121929
75%          0.599541
max          5.133557
Name: theta_all, dtype: float64

Missing theta values:
0

Infinite theta values:
0


## Fit the first-encounter-only IRT model

In [ ]:
models_first = models_first.to(device)
items_first = items_first.to(device)
responses_first = responses_first.to(device)

pyro.clear_param_store()

irt_model_first = OneParamLog(
    priors="hierarchical",
    device=device,
    num_items=len(problem_ids_first),
    num_models=len(student_ids_first),
    verbose=True
)

NUM_EPOCHS = 1000

irt_model_first.fit(
    models=models_first,
    items=items_first,
    responses=responses_first,
    num_epochs=NUM_EPOCHS
)

[epoch 0001] loss: 58355415.5546
[epoch 0101] loss: 7045858.6059
[epoch 0201] loss: 6849517.8473
[epoch 0301] loss: 6805723.3638
[epoch 0401] loss: 7364404.0722
[epoch 0501] loss: 7163436.9552
[epoch 0601] loss: 7632380.8338
[epoch 0701] loss: 6865326.9690
[epoch 0801] loss: 7069711.3326
[epoch 0901] loss: 7131107.6732
[epoch 1000] loss: 6830665.5939


## Extract first-encounter-only ability estimates

In [ ]:
param_store_first = pyro.get_param_store()

theta_first = (
    param_store_first["loc_ability"]
    .detach()
    .cpu()
    .numpy()
)

irt_scores_first = pd.DataFrame({
    "uuid": student_ids_first,
    "theta_first": theta_first
})

print("Number of students:", len(irt_scores_first))
print("\nTheta summary:")
print(irt_scores_first["theta_first"].describe())

print("\nMissing theta values:")
print(irt_scores_first["theta_first"].isna().sum())

print("\nInfinite theta values:")
print(np.isinf(irt_scores_first["theta_first"]).sum())

Number of students: 72758

Theta summary:
count    72758.000000
mean         0.482927
std          1.491703
min         -7.991844
25%         -0.323626
50%          0.421152
75%          1.226698
max          7.575851
Name: theta_first, dtype: float64

Missing theta values:
0

Infinite theta values:
0


## Run correlation comparison

In [ ]:
from scipy.stats import pearsonr, spearmanr

theta_comparison = pd.merge(
    irt_scores_all,
    irt_scores_first,
    on="uuid",
    how="inner"
)

pearson_r, pearson_p = pearsonr(
    theta_comparison["theta_all"],
    theta_comparison["theta_first"]
)

spearman_rho, spearman_p = spearmanr(
    theta_comparison["theta_all"],
    theta_comparison["theta_first"]
)

print("Students compared:", len(theta_comparison))

print("\nPearson correlation:")
print("r =", pearson_r)
print("p =", pearson_p)

print("\nSpearman correlation:")
print("rho =", spearman_rho)
print("p =", spearman_p)

Students compared: 72758

Pearson correlation:
r = 0.9495606
p = 0.0

Spearman correlation:
rho = 0.9827773561482485
p = 0.0


## Create ability groups from the primary all-encounter IRT estimates

In [ ]:
# 1. Equal-sized quartiles
irt_scores_all["theta_quartile"] = pd.qcut(
    irt_scores_all["theta_all"],
    q=4,
    labels=["Q1", "Q2", "Q3", "Q4"]
)

# 2. Standard-deviation-based groups
theta_sd = irt_scores_all["theta_all"].std()

sd_multipliers = np.array([
    -np.inf, -1.5, -1.0, -0.5,
     0.0, 0.5, 1.0, 1.5, np.inf
])

theta_sd_boundaries = sd_multipliers * theta_sd

sd_labels = [
    "< -1.5 SD",
    "-1.5 to -1.0 SD",
    "-1.0 to -0.5 SD",
    "-0.5 to 0 SD",
    "0 to +0.5 SD",
    "+0.5 to +1.0 SD",
    "+1.0 to +1.5 SD",
    "> +1.5 SD"
]

irt_scores_all["theta_sd_group"] = pd.cut(
    irt_scores_all["theta_all"],
    bins=theta_sd_boundaries,
    labels=sd_labels,
    include_lowest=True
)

print("Theta SD:", theta_sd)

print("\nQuartile counts:")
print(irt_scores_all["theta_quartile"].value_counts().sort_index())

print("\nSD-group counts:")
print(irt_scores_all["theta_sd_group"].value_counts().sort_index())

Theta SD: 1.1475660800933838

Quartile counts:
theta_quartile
Q1    18190
Q2    18189
Q3    18189
Q4    18190
Name: count, dtype: int64

SD-group counts:
theta_sd_group
< -1.5 SD           5423
-1.5 to -1.0 SD     7060
-1.0 to -0.5 SD    11968
-0.5 to 0 SD       15274
0 to +0.5 SD       14323
+0.5 to +1.0 SD     9692
+1.0 to +1.5 SD     5169
> +1.5 SD           3849
Name: count, dtype: int64


## Save primary IRT ability estimates and grouping assignments

In [ ]:
ability_groups = irt_scores_all[
    [
        "uuid",
        "theta_all",
        "theta_quartile",
        "theta_sd_group"
    ]
].copy()

ability_groups.to_csv(
    ABILITY_GROUP_PATH,
    index=False
)

print("Saved:", ABILITY_GROUP_PATH)
print("Shape:", ability_groups.shape)
print("\nColumns:")
print(ability_groups.columns.tolist())

Saved: /content/drive/MyDrive/education-ml-research/Junyi/student_irt_ability_groups.csv
Shape: (72758, 4)

Columns:
['uuid', 'theta_all', 'theta_quartile', 'theta_sd_group']
